# Comparación de Modelos Númericos con puntos alazar

A continución, se muestra la comparación de varios modelos númericos considerando 15 puntos alazar para las siguiente funciones:
- Función Esfera (10 variables)
- Función McCormick (2 variables)
- Función de Dixon-Price (4 variables)
- Función de Easom (2 variables)
- Función de Rastrigin (5 variables)

NOTA:
-  Las funciones y los modelos fueron implementados como modulos individuales para poder ser invocados desde el notebook

## Importación de Dependencias y Generación de puntos aleatorios

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

In [4]:
# Dependencias

import numpy as np
import random
from config.logging import setUpLogging
from graph.graph_foo import FunctionPlotter2D
from numerical_methods.gradient_descent import MDG_Wolfe
from numerical_methods.newton import NewtonMethod
from numerical_methods.max_decrease import MDG_DM
from functions.sphere_foo import SphereFunction
from functions.mcCormick_foo import McCormickFunction
from functions.dixon_price_foo import DixonPriceFunction
from functions.easom_foo import EasomFunction
from functions.rastrigin_foo import RastriginFunction
from utils.aux_functions import getTestValues, mergeLists, crear_tabla_comparativa
import logging

setUpLogging()
logger = logging.getLogger(__name__)

# Cantidad de puntos
n_points = 15

# Función Esfera con 10 variables
sphere = SphereFunction()
test_data_sphere: np.array = getTestValues(n_points, 2, sphere.domain[0], sphere.domain[1])
print("Sphere Test Data: ", test_data_sphere)


# Función McCormick con 2 variables
mcCormick = McCormickFunction()

test_data_mcCormick_x0: np.array = getTestValues(n_points, 2, mcCormick.domain[0], mcCormick.domain[1])
test_data_mcCormick_x1: np.array = getTestValues(n_points, 2, mcCormick.domain_x2[0], mcCormick.domain_x2[1])
test_data_mcCormick: np.array[tuple] = mergeLists(test_data_mcCormick_x0, test_data_mcCormick_x1)
print("McCormick Test Data: ",test_data_mcCormick)

# Función de Dixon-Price con 4 variables
dixon_price = DixonPriceFunction()

test_data_dixon_price: np.array = getTestValues(n_points, 4, dixon_price.domain[0], dixon_price.domain[1])
print("Dixon-Price Test Data: ", test_data_dixon_price)

# Función de Easom con 2 variables
easom = EasomFunction()

test_data_easom: np.array = getTestValues(n_points, 2, easom.domain[0], easom.domain[1])
print("Easom Test Data: ", test_data_easom)

# Función de Rastrigin con 5 variables
rastrigin = RastriginFunction()

test_data_rastrigin: np.array = getTestValues(n_points, 5, rastrigin.domain[0], rastrigin.domain[1])
print("Rastrigin Test Data: ", test_data_rastrigin)



Sphere Test Data:  [[-0.04912917  1.22119095]
 [ 3.28590704 -5.01554257]
 [ 0.68830746 -3.61836944]
 [-0.9217672   4.8143876 ]
 [-3.92249617 -2.96743859]
 [ 1.96533849  2.274301  ]
 [-0.86925087 -2.14847109]
 [-0.64362064 -3.2991366 ]
 [ 0.21820641  4.78244268]
 [ 4.72013325 -2.86857405]
 [ 4.79429733 -1.37979139]
 [-3.03413488 -2.53367855]
 [ 2.98954361 -1.52994177]
 [ 0.39306957  2.75780297]
 [ 0.50268464  3.29925749]]
McCormick Test Data:  [[-0.26216569  0.57621342]
 [ 3.36943631  1.86283426]
 [ 0.17849796 -0.60778796]
 [ 3.72704906 -1.95197074]
 [-1.27308517  0.32333359]
 [-1.07234125 -1.12803598]
 [ 3.33893668  2.71418884]
 [-1.08289881 -1.79208545]
 [ 2.44683632  1.25991356]
 [ 2.54316352 -2.856705  ]
 [ 0.59047159 -0.32777377]
 [ 1.42534088 -0.49266657]
 [-1.00173998  3.40678415]
 [-1.41290226 -2.11641656]
 [ 2.94699778  0.82276593]
 [ 2.06769409  3.86104874]
 [ 2.7384462   0.46724006]
 [ 2.76688183 -1.39111147]
 [ 2.8288733   0.45517535]
 [ 3.50975218  3.53816949]
 [ 2.59672949

# Comparación: Función Esfera (10 variables)

In [5]:

# Descenso del Gradiente con condiciones fuertes de Wolfe

resultados_sphere_MDG_Wolfe = {}
resultados_sphere_Newton = {}
resultados_sphere_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_sphere):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = sphere.f,
        Df       = sphere.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_sphere_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        # "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": sphere.f_count_invok,
        "Df_invok": sphere.Df_count_invok,
        "H_invok": sphere.H_count_invok
    }

    sphere.f_count_invok = 0
    sphere.Df_count_invok = 0
    sphere.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=sphere.Df,
        H=sphere.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_sphere_Newton[i] = {
        "x_optimo": x_opt_2,
        # "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": sphere.f_count_invok,
        "Df_invok": sphere.Df_count_invok,
        "H_invok": sphere.H_count_invok
    }

    sphere.f_count_invok = 0
    sphere.Df_count_invok = 0
    sphere.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=sphere.f,
        Df=sphere.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_sphere_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        # "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": sphere.f_count_invok,
        "Df_invok": sphere.Df_count_invok,
        "H_invok": sphere.H_count_invok
    }

    sphere.f_count_invok = 0
    sphere.Df_count_invok = 0
    sphere.H_count_invok = 0



# Comparación: Función McCormick (2 variables)

In [6]:
# Descenso del Gradiente con condiciones fuertes de Wolfe

resultados_mcCormick_MDG_Wolfe = {}
resultados_mcCormick_Newton = {}
resultados_mcCormick_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_mcCormick):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = mcCormick.f,
        Df       = mcCormick.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_mcCormick_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        # "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": mcCormick.f_count_invok,
        "Df_invok": mcCormick.Df_count_invok,
        "H_invok": mcCormick.H_count_invok
    }

    mcCormick.f_count_invok = 0
    mcCormick.Df_count_invok = 0
    mcCormick.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=mcCormick.Df,
        H=mcCormick.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_mcCormick_Newton[i] = {
        "x_optimo": x_opt_2,
        # "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": mcCormick.f_count_invok,
        "Df_invok": mcCormick.Df_count_invok,
        "H_invok": mcCormick.H_count_invok
    }

    mcCormick.f_count_invok = 0
    mcCormick.Df_count_invok = 0
    mcCormick.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=mcCormick.f,
        Df=mcCormick.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_mcCormick_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        # "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": mcCormick.f_count_invok,
        "Df_invok": mcCormick.Df_count_invok,
        "H_invok": mcCormick.H_count_invok
    }

    mcCormick.f_count_invok = 0
    mcCormick.Df_count_invok = 0
    mcCormick.H_count_invok = 0


# Comparación: Función de Dixon-Price (4 variables)

In [7]:

resultados_dixon_price_MDG_Wolfe = {}
resultados_dixon_price_Newton = {}
resultados_dixon_price_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_dixon_price):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = dixon_price.f,
        Df       = dixon_price.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_dixon_price_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        # "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": dixon_price.f_count_invok,
        "Df_invok": dixon_price.Df_count_invok,
        "H_invok": dixon_price.H_count_invok
    }

    dixon_price.f_count_invok = 0
    dixon_price.Df_count_invok = 0
    dixon_price.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=dixon_price.Df,
        H=dixon_price.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_dixon_price_Newton[i] = {
        "x_optimo": x_opt_2,
        # "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": dixon_price.f_count_invok,
        "Df_invok": dixon_price.Df_count_invok,
        "H_invok": dixon_price.H_count_invok
    }

    dixon_price.f_count_invok = 0
    dixon_price.Df_count_invok = 0
    dixon_price.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=dixon_price.f,
        Df=dixon_price.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_dixon_price_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        # "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": dixon_price.f_count_invok,
        "Df_invok": dixon_price.Df_count_invok,
        "H_invok": dixon_price.H_count_invok
    }

    dixon_price.f_count_invok = 0
    dixon_price.Df_count_invok = 0
    dixon_price.H_count_invok = 0


# Comparación: Función de Easom (2 variables)

In [8]:

resultados_easom_MDG_Wolfe = {}
resultados_easom_Newton = {}
resultados_easom_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_easom):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = easom.f,
        Df       = easom.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_easom_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        # "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": easom.f_count_invok,
        "Df_invok": easom.Df_count_invok,
        "H_invok": easom.H_count_invok
    }

    easom.f_count_invok = 0
    easom.Df_count_invok = 0
    easom.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=easom.Df,
        H=easom.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_easom_Newton[i] = {
        "x_optimo": x_opt_2,
        # "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": easom.f_count_invok,
        "Df_invok": easom.Df_count_invok,
        "H_invok": easom.H_count_invok
    }

    easom.f_count_invok = 0
    easom.Df_count_invok = 0
    easom.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=easom.f,
        Df=easom.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_easom_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        # "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": easom.f_count_invok,
        "Df_invok": easom.Df_count_invok,
        "H_invok": easom.H_count_invok
    }

    easom.f_count_invok = 0
    easom.Df_count_invok = 0
    easom.H_count_invok = 0

/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/numerical_methods/max_decrease.py:20: RuntimeWarning: invalid value encountered in scalar divide
  xk1 = (df(xk1)*xk - df(xk)*xk1) / (df(xk1) - df(xk))
2026-05-10 11:19:36 | ERROR | numerical_methods.newton | Valores provacan un Matrix Singular
/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/numerical_methods/max_decrease.py:20: RuntimeWarning: divide by zero encountered in scalar divide
  xk1 = (df(xk1)*xk - df(xk)*xk1) / (df(xk1) - df(xk))
/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/functions/easom_foo.py:27: RuntimeWarning: invalid value encountered in cos
  return -1*cos(x[0])*cos(x[1])*exp(-1*(x[0] - pi)**2 - (x[1] - pi)**2)
/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/functions/easom_foo.py:72: RuntimeWarning: invalid value encountered in cos
  cos(x2) * E * (
/Users/mardnstuff/Documents/dev/MARDGithub/mard-numerical-methods/functions/easom_foo.py:73: Run

# Comparación: Función de Rastrigin (5 variables)

In [ ]:

resultados_rastrigin_MDG_Wolfe = {}
resultados_rastrigin_Newton = {}
resultados_rastrigin_MDG_DM = {}

n_iter = 20
for i, test in enumerate(test_data_rastrigin):

    x_opt_1, trayectoria_1 = MDG_Wolfe(
        f        = rastrigin.f,
        Df       = rastrigin.Df,
        x0       = test,
        c1       = 1e-4,    # qué tan estricto es el descenso (Armijo)
        c2       = 0.9,     # qué tan estricta es la curvatura
        alph_max = 1.0,     # paso máximo que puede probar la búsqueda
        max_iter = n_iter
    )

    resultados_rastrigin_MDG_Wolfe[i] = {
        "x_optimo": x_opt_1,
        # "trayectoria": trayectoria_1,
        "n_iter": n_iter,
        "f_invok": rastrigin.f_count_invok,
        "Df_invok": rastrigin.Df_count_invok,
        "H_invok": rastrigin.H_count_invok
    }

    rastrigin.f_count_invok = 0
    rastrigin.Df_count_invok = 0
    rastrigin.H_count_invok = 0

    x_opt_2, trayectoria_2 = NewtonMethod(
        G=rastrigin.Df,
        H=rastrigin.H,
        x0=test,
        max_iter=n_iter
    )

    resultados_rastrigin_Newton[i] = {
        "x_optimo": x_opt_2,
        # "trayectoria": trayectoria_2,
        "n_iter": n_iter,
        "f_invok": rastrigin.f_count_invok,
        "Df_invok": rastrigin.Df_count_invok,
        "H_invok": rastrigin.H_count_invok
    }

    rastrigin.f_count_invok = 0
    rastrigin.Df_count_invok = 0
    rastrigin.H_count_invok = 0

    x_opt_3, trayectoria_3 = MDG_DM(
        f=rastrigin.f,
        Df=rastrigin.Df,
        x0=test,
        max_iter=n_iter
    )

    resultados_rastrigin_MDG_DM[i] = {
        "x_optimo": x_opt_3,
        # "trayectoria": trayectoria_3,
        "n_iter": n_iter,
        "f_invok": rastrigin.f_count_invok,
        "Df_invok": rastrigin.Df_count_invok,
        "H_invok": rastrigin.H_count_invok
    }

    rastrigin.f_count_invok = 0
    rastrigin.Df_count_invok = 0
    rastrigin.H_count_invok = 0

tabla_rastrigin = crear_tabla_comparativa(
    resultados_por_metodo={
        "MDG_Wolfe": resultados_rastrigin_MDG_Wolfe,
        "Newton":    resultados_rastrigin_Newton,
        "MDG_DM":    resultados_rastrigin_MDG_DM,
    },
    nombre_funcion="Rastrigin"
)
# TODO: Hacer una tabla comparativa que contenga todo y promedios

display(tabla_rastrigin)  # display() renderiza mejor que print en notebooks



📊 Tabla comparativa — Rastrigin


,Punto,Método,x_optimo,n_iter,f_invok,Df_invok,H_invok
0,1,MDG_DM,"[1.3455, -10.4463, 7.9367, -7.4853, 1.6648]",20,2456,2476,0
1,1,MDG_Wolfe,"[0.9865, 2.9822, -1.9926, -2.9791, -1.9864]",20,881,47,0
2,1,Newton,"[0.0, 4.523, -4.523, -3.5179, -1.9899]",20,0,4,4
3,2,MDG_DM,"[-1.2065, 3.8453, -1.8774, 5.0224, -15.0754]",20,2288,2308,0
4,2,MDG_Wolfe,"[0.9939, 7.9585, 4.9754, -0.0016, -2.9852]",20,642,50,0
5,2,Newton,"[3.9798, 7.5386, 4.523, -4.9747, -3.5179]",20,0,7,7
6,3,MDG_DM,"[5.7929, 3.997, -12.7628, -1.666, 16.02]",20,2224,2244,0
7,3,MDG_Wolfe,"[0.0027, -1.9877, -0.0031, 0.9909, 0.9951]",20,680,56,0
8,3,Newton,"[2.9849, 4.523, -7.9592, -0.5025, -1.5076]",20,0,6,6
9,4,MDG_DM,"[0.7943, 11.2701, 7.9014, -14.0163, -12.3725]",20,2448,2468,0
